# Tasas de plazo fijo — Argentina

El BCRA publica diariamente el promedio ponderado de la tasa
del plazo fijo a 30 días. Es un benchmark de mercado, sin tratarse de un promedio simple, sino ponderado por volumen.

In [1]:
import requests
import pandas as pd
import sqlite3 
from datetime import date

url = "https://api.bcra.gob.ar/estadisticas/v4.0/Monetarias/12"

response = requests.get(url, timeout=10)
response.raise_for_status()
data_monetaria = response.json()

In [2]:
print(response.status_code) #verifico que la API sea la correcta 

200


In [3]:
print(data_monetaria.keys())

dict_keys(['status', 'metadata', 'results'])


In [4]:
print(data_monetaria["results"])

[{'idVariable': 12, 'detalle': [{'fecha': '2026-03-27', 'valor': 23.9}, {'fecha': '2026-03-26', 'valor': 25.1}, {'fecha': '2026-03-25', 'valor': 24.88}, {'fecha': '2026-03-20', 'valor': 24.97}, {'fecha': '2026-03-19', 'valor': 25.95}, {'fecha': '2026-03-18', 'valor': 26.31}, {'fecha': '2026-03-17', 'valor': 27.18}, {'fecha': '2026-03-16', 'valor': 27.39}, {'fecha': '2026-03-13', 'valor': 27.16}, {'fecha': '2026-03-12', 'valor': 27.81}, {'fecha': '2026-03-11', 'valor': 27.28}, {'fecha': '2026-03-10', 'valor': 28.27}, {'fecha': '2026-03-09', 'valor': 27.66}, {'fecha': '2026-03-06', 'valor': 27.5}, {'fecha': '2026-03-05', 'valor': 27.27}, {'fecha': '2026-03-04', 'valor': 27.37}, {'fecha': '2026-03-03', 'valor': 27.88}, {'fecha': '2026-03-02', 'valor': 28.08}, {'fecha': '2026-02-27', 'valor': 28.28}, {'fecha': '2026-02-26', 'valor': 29.12}, {'fecha': '2026-02-25', 'valor': 28.04}, {'fecha': '2026-02-24', 'valor': 31.82}, {'fecha': '2026-02-23', 'valor': 30.16}, {'fecha': '2026-02-20', 'val

El BCRA publica el promedio del plazo fijo con un `día de rezago`. 
EL primer valor refleja la tasa del día de ayer (d-1). 

In [5]:
print(len(data_monetaria["results"])) #q de elementos de results  

1


In [6]:
data_monetaria["results"][0]["detalle"]

[{'fecha': '2026-03-27', 'valor': 23.9},
 {'fecha': '2026-03-26', 'valor': 25.1},
 {'fecha': '2026-03-25', 'valor': 24.88},
 {'fecha': '2026-03-20', 'valor': 24.97},
 {'fecha': '2026-03-19', 'valor': 25.95},
 {'fecha': '2026-03-18', 'valor': 26.31},
 {'fecha': '2026-03-17', 'valor': 27.18},
 {'fecha': '2026-03-16', 'valor': 27.39},
 {'fecha': '2026-03-13', 'valor': 27.16},
 {'fecha': '2026-03-12', 'valor': 27.81},
 {'fecha': '2026-03-11', 'valor': 27.28},
 {'fecha': '2026-03-10', 'valor': 28.27},
 {'fecha': '2026-03-09', 'valor': 27.66},
 {'fecha': '2026-03-06', 'valor': 27.5},
 {'fecha': '2026-03-05', 'valor': 27.27},
 {'fecha': '2026-03-04', 'valor': 27.37},
 {'fecha': '2026-03-03', 'valor': 27.88},
 {'fecha': '2026-03-02', 'valor': 28.08},
 {'fecha': '2026-02-27', 'valor': 28.28},
 {'fecha': '2026-02-26', 'valor': 29.12},
 {'fecha': '2026-02-25', 'valor': 28.04},
 {'fecha': '2026-02-24', 'valor': 31.82},
 {'fecha': '2026-02-23', 'valor': 30.16},
 {'fecha': '2026-02-20', 'valor': 30.

In [7]:
df_hist = pd.DataFrame(data_monetaria["results"][0]["detalle"]) 
print(df_hist.head())  

        fecha  valor
0  2026-03-27  23.90
1  2026-03-26  25.10
2  2026-03-25  24.88
3  2026-03-20  24.97
4  2026-03-19  25.95


## Tasas por banco

El BCRA tiene un comparador público de tasas de plazos fijos, 
pero no tiene API documentada. Al inspeccionar el tráfico de red se puede 
acceder directamente al endpoint que alimenta la página.

In [8]:
url_bancos = "https://www.bcra.gob.ar/api-plazos-fijos.php"

response_bancos = requests.get(url_bancos, timeout=10)
response_bancos.raise_for_status()
data_bancos = response_bancos.json()

print(data_bancos.keys()) 

dict_keys(['success', 'top10', 'otros'])


In [9]:
df_top10 = pd.DataFrame(data_bancos["top10"])
df_otros = pd.DataFrame(data_bancos["otros"])
df_bancos = pd.concat([df_top10, df_otros], ignore_index=True) 

df_bancos.head() 

,codigo,entidad,logo_url,tasa_con_relacion,tasa_sin_relacion,web
0,11,BANCO DE LA NACION ARGENTINA ...,http://www.bcra.gob.ar/archivos/Imagenes/logos...,22.0,0.0,
1,72,BANCO SANTANDER ARGENTINA S.A. ...,http://www.bcra.gob.ar/archivos/Imagenes/logos...,20.0,0.0,
2,7,BANCO DE GALICIA Y BUENOS AIRES S.A. ...,http://www.bcra.gob.ar/archivos/Imagenes/logos...,21.0,0.0,
3,14,BANCO DE LA PROVINCIA DE BUENOS AIRES ...,http://www.bcra.gob.ar/archivos/Imagenes/logos...,23.0,0.0,
4,17,BANCO BBVA ARGENTINA S.A. ...,http://www.bcra.gob.ar/archivos/Imagenes/logos...,21.0,0.0,


## Fecha de los datos

La API utilizada para obtener las tasas de plazo fijo por banco no incluye
un campo de "fecha".

Por esta razón se agrega una columna `fecha` que representa la fecha de
extracción de los datos.

Las tasas obtenidas corresponden a las tasas vigentes al momento de
consultar la API.

In [10]:
df_bancos = df_bancos[["codigo","entidad","tasa_con_relacion"]]

df_bancos = df_bancos.rename(columns={
    "entidad":"banco",
    "tasa_con_relacion":"tasa_pf_30d"
})

fecha_hoy = date.today() 
df_bancos["fecha"] = fecha_hoy

df_bancos.head()  

,codigo,banco,tasa_pf_30d,fecha
0,11,BANCO DE LA NACION ARGENTINA ...,22.0,2026-03-31
1,72,BANCO SANTANDER ARGENTINA S.A. ...,20.0,2026-03-31
2,7,BANCO DE GALICIA Y BUENOS AIRES S.A. ...,21.0,2026-03-31
3,14,BANCO DE LA PROVINCIA DE BUENOS AIRES ...,23.0,2026-03-31
4,17,BANCO BBVA ARGENTINA S.A. ...,21.0,2026-03-31


## Spread

El *spread* se define como la diferencia entre la tasa de plazo fijo ofrecida por cada banco y el promedio ponderado del sistema financiero publicado por el BCRA.

La comparación se realiza entre las tasas vigentes y el promedio del mercado del día anterior.

In [11]:
data_monetaria.keys()

dict_keys(['status', 'metadata', 'results'])

In [12]:
valor_promedio = data_monetaria["results"][0]["detalle"][0]["valor"]

df_bancos["promedio_bcra"] = valor_promedio 
df_bancos["spread"] = df_bancos["tasa_pf_30d"] - df_bancos["promedio_bcra"]

df_bancos.head()

,codigo,banco,tasa_pf_30d,fecha,promedio_bcra,spread
0,11,BANCO DE LA NACION ARGENTINA ...,22.0,2026-03-31,23.9,-1.9
1,72,BANCO SANTANDER ARGENTINA S.A. ...,20.0,2026-03-31,23.9,-3.9
2,7,BANCO DE GALICIA Y BUENOS AIRES S.A. ...,21.0,2026-03-31,23.9,-2.9
3,14,BANCO DE LA PROVINCIA DE BUENOS AIRES ...,23.0,2026-03-31,23.9,-0.9
4,17,BANCO BBVA ARGENTINA S.A. ...,21.0,2026-03-31,23.9,-2.9


In [13]:
df_bancos['banco'] = df_bancos['banco'].str.strip().str.upper()

conn = sqlite3.connect("../tasas_bancos.db") 
 
tabla_existe = conn.execute("""
    SELECT name FROM sqlite_master 
    WHERE type='table' AND name='tasas_bancos'
""").fetchone()

if tabla_existe:
    conn.execute("DELETE FROM tasas_bancos WHERE fecha = ?", (fecha_hoy.isoformat(),))
    conn.commit()

df_bancos.to_sql(
    "tasas_bancos",
    conn,
    if_exists="append",
    index=False
)

conn.commit()
conn.close() 